# CNN-LSTM Model Training

In [0]:
%run ../config/config

In [0]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as Fn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import mlflow
import mlflow.pytorch
from pyspark.sql import functions as F
from sklearn.metrics import f1_score, accuracy_score, classification_report

#### 1. Load flat data & create windows

In [0]:
ml_features_df = spark.table(tables["ml_features"])

feature_cols = [c for c in ml_feature_columns if c in ml_features_df.columns]
print(f"Feature columns: {len(feature_cols)}")

tmp_spark = "/tmp/wyckoff_ml_features_07"   # dbfs:/tmp/...
tmp_local = "/dbfs/tmp/wyckoff_ml_features_07"  # pandas reads FUSE mount
dbutils.fs.rm(tmp_spark, recurse=True)

for split in ["train", "val", "test"]:
    ml_features_df.filter(F.col("split") == split) \
        .write.mode("overwrite").parquet(f"{tmp_spark}/{split}")
    print(f"  Written {split} parquet")

train_pdf = pd.read_parquet(f"{tmp_local}/train").sort_values(["symbol", "date"]).reset_index(drop=True)
val_pdf = pd.read_parquet(f"{tmp_local}/val").sort_values(["symbol", "date"]).reset_index(drop=True)
test_pdf = pd.read_parquet(f"{tmp_local}/test").sort_values(["symbol", "date"]).reset_index(drop=True)

dbutils.fs.rm(tmp_spark, recurse=True)
print(f"  Train: {len(train_pdf):,}  Val: {len(val_pdf):,}  Test: {len(test_pdf):,}")

In [0]:
def create_windows(pdf, feature_cols, window_size, stride, pl_threshold):
    """
    Create sliding windows from flat per-day data.
    """
    n_feat = len(feature_cols)

    # upper-bound estimate for pre-allocation
    max_windows = sum(
        max(0, (len(g) - window_size) // stride + 1)
        for _, g in pdf.groupby("symbol", sort=False)
    )
    if max_windows == 0:
        return np.array([]), np.array([]), [], []

    features = np.empty((max_windows, window_size, n_feat), dtype=np.float32)
    labels = np.empty(max_windows, dtype=np.int64)
    symbols_out, dates_out = [], []
    idx = 0

    for sym, group in pdf.groupby("symbol", sort=False):
        group = group.sort_values("date").reset_index(drop=True)
        n = len(group)
        if n < window_size:
            continue

        feat = group[feature_cols].values.astype(np.float32)
        feat = np.where(np.isinf(feat), np.nan, feat)
        # forward fill then fill remaining NaN with 0
        feat_df = pd.DataFrame(feat)
        feat = feat_df.ffill().fillna(0.0).values

        labs = group["phase_label"].values
        dts = group["date"].values
        pl = group["price_limit_hit"].values if "price_limit_hit" in group.columns else np.zeros(n)

        for i in range(window_size - 1, n, stride):
            s = i - window_size + 1

            # Skip if too many price-limit-hit days in window
            if np.nansum(pl[s:i+1]) / window_size > pl_threshold:
                continue

            window = feat[s:i+1]

            # Per-window z-score normalization
            mu = np.nanmean(window, axis=0, keepdims=True)
            sigma = np.nanstd(window, axis=0, keepdims=True)
            sigma = np.where(sigma == 0, 1.0, sigma)
            features[idx] = (window - mu) / sigma
            np.nan_to_num(features[idx], copy=False, nan=0.0, posinf=0.0, neginf=0.0)

            labels[idx] = int(labs[i])
            symbols_out.append(sym)
            dates_out.append(dts[i])
            idx += 1

    return features[:idx], labels[:idx], symbols_out, dates_out


print("Create sliding windows")
train_features, train_labels, _, _ = create_windows(
    train_pdf, feature_cols, window_size, window_stride, price_limit_threshold
)
del train_pdf
print(f"  Train windows: {len(train_labels):,}")

val_features, val_labels, _, _ = create_windows(
    val_pdf, feature_cols, window_size, window_stride, price_limit_threshold
)
del val_pdf
print(f"  Val windows: {len(val_labels):,}")

test_features, test_labels, test_symbols, test_dates = create_windows(
    test_pdf, feature_cols, window_size, window_stride, price_limit_threshold
)
del test_pdf
print(f"  Test windows:  {len(test_labels):,}")

if len(train_features) == 0:
    raise ValueError("Training set is empty")
if len(val_features) == 0:
    raise ValueError("Validation set is empty")

print(f"\nWindow shape: {train_features.shape[1]} × {train_features.shape[2]}  (expected: {window_size} × {len(feature_cols)})")
print(f"NaN: {np.isnan(train_features).sum()}")
print(f"Inf: {np.isinf(train_features).sum()}")

#### Data augmentation for rare classes (C, D)

In [0]:
cfg = CNNLSTMConfig()

if cfg.augment_rare_classes and len(train_features) > 0:
    from collections import Counter

    class_counts = Counter(train_labels.tolist())
    target_count = cfg.augment_max_target

    rare_classes = [cls for cls, cnt in class_counts.items() if cnt < target_count]
    print(f"Class distribution before augmentation:")
    for cls in sorted(class_counts):
        marker = " RARE " if cls in rare_classes else ""
        print(f"  Class {cls}: {class_counts[cls]:,}{marker}")
    print(f"Augment target cap: {target_count:,}")

    aug_features_list = []
    aug_labels_list = []

    rng = np.random.RandomState(42)

    for cls in rare_classes:
        cls_mask = train_labels == cls
        cls_features = train_features[cls_mask]
        n_existing = len(cls_features)
        n_needed = target_count - n_existing
        if n_needed <= 0:
            continue

        indices = rng.choice(n_existing, size=n_needed, replace=True)
        sampled = cls_features[indices].copy()

        noise = rng.normal(0, cfg.augment_jitter_std, size=sampled.shape).astype(np.float32)
        sampled = sampled + noise

        aug_features_list.append(sampled)
        aug_labels_list.append(np.full(n_needed, cls, dtype=np.int64))
        print(f"  Augmented class {cls}: {n_existing:,} so {n_existing + n_needed:,} (+{n_needed:,})")

    if aug_features_list:
        train_features = np.concatenate([train_features] + aug_features_list, axis=0)
        train_labels = np.concatenate([train_labels] + aug_labels_list, axis=0)

        # Shuffle augmented data into training set
        shuffle_idx = rng.permutation(len(train_labels))
        train_features = train_features[shuffle_idx]
        train_labels = train_labels[shuffle_idx]

        print(f"\nAfter augmentation: {len(train_labels):,} train windows")
    else:
        print("No classes needed augmentation.")
else:
    print("Augmentation disabled or no training data.")


#### 2. PyTorch Dataset

In [0]:
class WyckoffDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.from_numpy(features)
        self.labels = torch.from_numpy(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


train_ds = WyckoffDataset(train_features, train_labels)
val_ds = WyckoffDataset(val_features, val_labels)
test_ds = WyckoffDataset(test_features, test_labels)

# infer actual feature dimensions
actual_window_size = train_ds.features.shape[1]
actual_num_features = train_ds.features.shape[2]
print(f"Feature shape: ({actual_window_size}, {actual_num_features})")

# compute class weights for imbalanced data
class_counts = np.bincount(train_ds.labels, minlength=number_of_phases)
total = class_counts.sum()
class_weights = total / (number_of_phases * class_counts.clip(min=1))
max_cw = CNNLSTMConfig().max_class_weight
class_weights = np.clip(class_weights, 1.0, max_cw)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
print(f"Class counts: {class_counts}")
print(f"Class weights (capped at {max_cw}): {class_weights.round(2)}")

#### 3. CNN-LSTM model architecture

In [0]:
class MultiScaleCNNBlock(nn.Module):
    """Multi-scale 1D CNN with different kernel sizes."""

    def __init__(self, in_channels, filters, kernel_size):
        super().__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, filters, kernel_size, padding=padding)
        self.bn1 = nn.BatchNorm1d(filters)
        self.conv2 = nn.Conv1d(filters, filters, kernel_size, padding=padding)
        self.bn2 = nn.BatchNorm1d(filters)

    def forward(self, x):
        x = Fn.relu(self.bn1(self.conv1(x)))
        x = Fn.relu(self.bn2(self.conv2(x)))
        return x


class WyckoffCNNLSTM(nn.Module):
    """
    Input:  (batch, window_size, num_features)
    Output: (batch, num_classes)
    """

    def __init__(self, config: CNNLSTMConfig):
        super().__init__()
        self.config = config

        # Multi-scale CNN branches
        self.cnn_branches = nn.ModuleList()
        for filters, kernel in zip(config.cnn_filters, config.cnn_kernels):
            self.cnn_branches.append(
                MultiScaleCNNBlock(config.num_features, filters, kernel)
            )

        total_cnn_out = sum(config.cnn_filters)
        self.cnn_bn = nn.BatchNorm1d(total_cnn_out)
        self.cnn_dropout = nn.Dropout(config.cnn_dropout)

        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            input_size=total_cnn_out,
            hidden_size=config.lstm_hidden,
            num_layers=config.lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=config.lstm_dropout if config.lstm_layers > 1 else 0,
        )
        lstm_out = config.lstm_hidden * 2  # bidirectional

        # Multi-Head Attention
        self.attention = nn.MultiheadAttention(
            embed_dim=lstm_out,
            num_heads=config.attention_heads,
            batch_first=True,
            dropout=config.attention_dropout,
        )
        self.layer_norm = nn.LayerNorm(lstm_out)

        # Classifier MLP
        self.classifier = nn.Sequential(
            nn.Linear(lstm_out, config.mlp_hidden),
            nn.BatchNorm1d(config.mlp_hidden),
            nn.ReLU(),
            nn.Dropout(config.mlp_dropout),
            nn.Linear(config.mlp_hidden, config.num_classes),
        )

    def forward(self, x):
        # x: (batch, seq_len, features)
        # Conv1d expects (batch, channels, seq_len)
        x_t = x.transpose(1, 2)

        # Multi-scale CNN
        cnn_outs = [branch(x_t) for branch in self.cnn_branches]
        x_cnn = torch.cat(cnn_outs, dim=1)  # (batch, total_cnn_out, seq_len)
        x_cnn = self.cnn_dropout(self.cnn_bn(x_cnn))

        # Back to (batch, seq_len, features) for LSTM
        x_cnn = x_cnn.transpose(1, 2)

        # BiLSTM
        lstm_out, _ = self.lstm(x_cnn)  # (batch, seq_len, lstm_hidden*2)

        # Multi-Head Self-Attention
        attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)
        attn_out = self.layer_norm(attn_out + lstm_out) 

        pooled = attn_out.mean(dim=1)  # (batch, lstm_hidden*2)

        # Classify
        logits = self.classifier(pooled)  # (batch, num_classes)
        return logits

#### 4. Training loop


In [0]:
def train_model(model, train_ds, val_ds, config, class_weights, device, generator=None):
    """Full training loop with MLflow logging."""

    train_loader = DataLoader(
        train_ds, batch_size=config.batch_size, shuffle=True, drop_last=True, generator=generator,
    )
    val_loader = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False)

    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=config.label_smoothing)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay
    )
    
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-6)

    best_val_f1 = 0.0
    patience_counter = 0
    best_state = None

    for epoch in range(1, config.max_epochs + 1):
        # Train 
        model.train()
        train_loss = 0.0
        epoch_train_preds, epoch_train_labels = [], []

        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip_norm)
            optimizer.step()

            train_loss += loss.item() * batch_x.size(0)
            epoch_train_preds.extend(logits.argmax(dim=1).cpu().numpy())
            epoch_train_labels.extend(batch_y.cpu().numpy())

        samples_seen = len(train_loader) * config.batch_size 
        train_loss /= samples_seen
        train_f1 = f1_score(epoch_train_labels, epoch_train_preds, average="macro", zero_division=0)
        train_acc = accuracy_score(epoch_train_labels, epoch_train_preds)

        # Validate 
        model.eval()
        val_loss = 0.0
        epoch_val_preds, epoch_val_labels = [], []

        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                logits = model(batch_x)
                loss = criterion(logits, batch_y)

                val_loss += loss.item() * batch_x.size(0)
                epoch_val_preds.extend(logits.argmax(dim=1).cpu().numpy())
                epoch_val_labels.extend(batch_y.cpu().numpy())

        val_loss /= len(val_ds)
        val_f1 = f1_score(epoch_val_labels, epoch_val_preds, average="macro", zero_division=0)
        val_acc = accuracy_score(epoch_val_labels, epoch_val_preds)

        # Step scheduler (plateau needs val_loss; cosine steps unconditionally)
        scheduler.step(metrics= val_loss)
        current_lr = optimizer.param_groups[0]["lr"]

        # Log to MLflow
        mlflow.log_metrics({
            "train_loss": train_loss,
            "train_f1": train_f1,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_f1": val_f1,
            "val_acc": val_acc,
            "lr": current_lr,
        }, step=epoch)

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d} | Train Loss: {train_loss:.4f}, F1: {train_f1:.4f} | Val Loss: {val_loss:.4f}, F1: {val_f1:.4f} | LR: {current_lr:.2e}")

        # Early stopping
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            patience_counter = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= config.early_stop_patience:
                print(f"\nEarly stopping at epoch {epoch}. Best val F1: {best_val_f1:.4f}")
                break

    # Restore best model
    if best_state:
        model.load_state_dict(best_state)
        model.to(device)

    return model, best_val_f1

#### 5. Train with MLflow

In [0]:
SEEDS = [7, 42, 126]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Ensemble seeds: {SEEDS}")

# Update config with actual feature count
model_config = CNNLSTMConfig()
model_config.num_features = actual_num_features
model_config.window_size = actual_window_size

# MLflow experiment
mlflow.set_experiment(model_config.experiment_name)

# Collect test probabilities from each seed for ensemble averaging
seed_test_probs = []
seed_test_f1s = []
seed_val_f1s = []
seed_models = []
seed_run_ids = []

with mlflow.start_run(run_name="cnn_lstm_v14_final") as parent_run:
    parent_run_id = parent_run.info.run_id
    mlflow.log_params({
        "ensemble_seeds": str(SEEDS),
        "n_seeds": len(SEEDS),
        "window_size": model_config.window_size,
        "num_features": model_config.num_features,
        "cnn_filters": str(model_config.cnn_filters),
        "cnn_kernels": str(model_config.cnn_kernels),
        "lstm_hidden": model_config.lstm_hidden,
        "lstm_layers": model_config.lstm_layers,
        "attention_heads": model_config.attention_heads,
        "batch_size": model_config.batch_size,
        "learning_rate": model_config.learning_rate,
        "weight_decay": model_config.weight_decay,
        "label_smoothing": model_config.label_smoothing,
        "max_class_weight": model_config.max_class_weight,
        "optimizer": "AdamW",
        "scheduler": model_config.scheduler,
        "train_samples": len(train_ds),
        "val_samples": len(val_ds),
        "test_samples": len(test_ds),
    })

    for seed_idx, seed in enumerate(SEEDS):
        print(f"SEED {seed_idx+1}/{len(SEEDS)}: {seed}")

        # Set seed 
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
            torch.backends.cudnn.deterministic = True
        
        loader_generator = torch.Generator()
        loader_generator.manual_seed(seed)

        model = WyckoffCNNLSTM(model_config).to(device)
        if seed_idx == 0:
            params_total = sum(p.numel() for p in model.parameters())
            params_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"Model parameters: {params_total:,} (trainable: {params_trainable:,})")
            mlflow.log_params({
                "params_total": params_total,
                "params_trainable": params_trainable,
            })

        with mlflow.start_run(run_name=f"seed_{seed}", nested=True) as child_run:
            mlflow.log_params({"seed": seed})

            # Train
            model, best_val_f1 = train_model(
                model, train_ds, val_ds, model_config, class_weights_tensor, device,
                generator=loader_generator,
            )

            # Evaluate on test set
            test_loader = DataLoader(test_ds, batch_size=model_config.batch_size, shuffle=False)
            model.eval()
            seed_preds_list = []
            seed_labels_list = []
            seed_probs_list = []

            with torch.no_grad():
                for batch_x, batch_y in test_loader:
                    batch_x = batch_x.to(device)
                    logits = model(batch_x)
                    batch_probs = Fn.softmax(logits, dim=1)
                    seed_preds_list.extend(logits.argmax(dim=1).cpu().numpy().tolist())
                    seed_labels_list.extend(batch_y.cpu().numpy().tolist())
                    seed_probs_list.extend(batch_probs.cpu().numpy())

            seed_probs_arr = np.array(seed_probs_list)
            seed_test_probs.append(seed_probs_arr)

            seed_f1 = f1_score(seed_labels_list, seed_preds_list, average="macro", zero_division=0)
            seed_wf1 = f1_score(seed_labels_list, seed_preds_list, average="weighted", zero_division=0)
            seed_acc = accuracy_score(seed_labels_list, seed_preds_list)

            mlflow.log_metrics({
                "test_f1_macro": seed_f1,
                "test_f1_weighted": seed_wf1,
                "test_accuracy": seed_acc,
                "best_val_f1": best_val_f1,
            })

            seed_test_f1s.append(seed_f1)
            seed_val_f1s.append(best_val_f1)
            seed_models.append(model)
            seed_run_ids.append(child_run.info.run_id)

            print(f"Seed {seed}: Test Macro F1 = {seed_f1:.4f}, Val F1 = {best_val_f1:.4f}")

    # average probabilities 
    print(f"Ensemble results ({len(SEEDS)} seeds)")

    ensemble_probs = np.mean(seed_test_probs, axis=0)  # (N, 10)
    ensemble_preds = ensemble_probs.argmax(axis=1)
    test_labels_arr = np.array(seed_labels_list)

    ens_f1 = f1_score(test_labels_arr, ensemble_preds, average="macro", zero_division=0)
    ens_wf1 = f1_score(test_labels_arr, ensemble_preds, average="weighted", zero_division=0)
    ens_acc = accuracy_score(test_labels_arr, ensemble_preds)

    # Log ensemble metrics on parent run
    mlflow.log_metrics({
        "ensemble_test_f1_macro": ens_f1,
        "ensemble_test_f1_weighted": ens_wf1,
        "ensemble_test_accuracy": ens_acc,
        "best_single_f1": max(seed_test_f1s),
        "mean_single_f1": float(np.mean(seed_test_f1s)),
        "f1_std": float(np.std(seed_test_f1s)),
    })

    print(f"\nIndividual seeds:")
    for seed, f1 in zip(SEEDS, seed_test_f1s):
        print(f"  Seed {seed}: {f1:.4f}")
    print(f"\nEnsemble (averaged softmax):")
    print(f"  Accuracy:    {ens_acc:.4f}")
    print(f"  Macro F1:    {ens_f1:.4f}")
    print(f"  Weighted F1: {ens_wf1:.4f}")
    print(f"  vs best single:  {ens_f1 - max(seed_test_f1s):+.4f}")
    print(f"  vs mean single:  {ens_f1 - np.mean(seed_test_f1s):+.4f}")

    print(f"\nClassification:")
    print(classification_report(
        test_labels_arr, ensemble_preds,
        target_names=[phase_names[i] for i in range(number_of_phases)],
        zero_division=0,
    ))

    # Set variables for next section
    run_id = parent_run_id
    test_preds = ensemble_preds
    test_probs = ensemble_probs.tolist()
    test_labels = test_labels_arr
    test_f1 = ens_f1
    test_wf1 = ens_wf1
    test_acc = ens_acc

    print(f"MLflow Parent Run ID: {run_id}")

In [0]:
from mlflow.models import infer_signature

best_seed_idx = int(np.argmax(seed_test_f1s))
best_model = seed_models[best_seed_idx]
best_seed = SEEDS[best_seed_idx]
print(f"Best single model: seed={best_seed}, F1={seed_test_f1s[best_seed_idx]:.4f}")

sample_input = test_ds.features[:2].numpy()
best_model.eval()
with torch.no_grad():
    sample_output = best_model(test_ds.features[:2].to(device)).cpu().numpy()
signature = infer_signature(sample_input, sample_output)

with mlflow.start_run(run_id=run_id):
    mlflow.pytorch.log_model(
        best_model,
        artifact_path="model",
        signature=signature,
        registered_model_name=f"{catalog}.{gold_schema}.{model_config.model_name}",
    )
print(f"Model logged to run {run_id}")

In [0]:
pred_rows = [
    (
        sym, dt, CNNLSTMConfig().model_name, run_id, int(pred), int(label),
        float(test_f1), float(test_wf1), float(test_acc),
        [float(p) for p in prob],
    )
    for sym, dt, pred, label, prob in zip(test_symbols, test_dates, test_preds, test_labels, test_probs)
]

pred_schema = "symbol STRING, date DATE, model_name STRING, run_id STRING, predicted INT, actual INT, macro_f1 DOUBLE, weighted_f1 DOUBLE, accuracy DOUBLE, probabilities ARRAY<DOUBLE>"
pred_table = f"{catalog}.{gold_schema}.cnn_lstm_predictions"
spark.createDataFrame(pred_rows, schema=pred_schema).write.mode("overwrite").saveAsTable(pred_table)
print(f"Ensemble predictions saved to {pred_table} ({len(pred_rows):,} rows)")

In [0]:
import mlflow

client = mlflow.MlflowClient()
model_name = f"{catalog}.{gold_schema}.wyckoff_cnn_lstm"

try:
    versions = client.search_model_versions(f"name='{model_name}'")
    
    for v in versions:
        print(f"Version: {v.version}, Status: {v.status}, Run ID: {v.run_id}")
    
    print(f"\nModel successfull registered: {len(versions)} version(s)")

except Exception as e:
    print(f"Model not found: {e}")

#### 6. Save test predictions for evaluation

In [0]:
pred_table = f"{catalog}.{gold_schema}.cnn_lstm_predictions"
pred_df = spark.table(pred_table)
print(f"Prediction rows: {pred_df.count():,}")
display(pred_df.groupBy("predicted").count().orderBy("predicted"))